In [1]:
import sympy as sp
from sympy.physics.quantum import TensorProduct
from basic_gates import *

### 📡 Quantum Teleportation Circuit (-Y, Y: $\ket{Msg}$, $\ket{A}$)

![Quantum Teleportation](../assets/QT_-YY.png)

> View the full circuit in [Quirk - （-Y, Y）](https://algassert.com/quirk#circuit=%7B%22cols%22%3A%5B%5B1%2C%22H%22%5D%2C%5B1%2C%22%E2%80%A2%22%2C1%2C1%2C%22X%22%5D%2C%5B%22%E2%80%A6%22%2C%22%E2%80%A6%22%2C1%2C1%2C%22%E2%80%A6%22%5D%2C%5B%22%E2%80%A6%22%2C%22%E2%80%A6%22%2C1%2C1%2C%22%E2%80%A6%22%5D%2C%5B%22~87lj%22%5D%2C%5B%22Bloch%22%5D%2C%5B%22%E2%80%A2%22%2C%22X%22%5D%2C%5B%22H%22%5D%2C%5B%22Z%5E%C2%BD%22%2C%22Z%5E-%C2%BD%22%5D%2C%5B%22H%22%2C%22H%22%5D%2C%5B%22Measure%22%2C%22Measure%22%5D%2C%5B1%2C%22%E2%80%A2%22%2C1%2C1%2C%22X%22%5D%2C%5B%22%E2%80%A2%22%2C1%2C1%2C1%2C%22Y%22%5D%2C%5B1%2C1%2C1%2C1%2C%22Z%5E%C2%BD%22%5D%2C%5B1%2C1%2C1%2C1%2C%22H%22%5D%2C%5B1%2C1%2C1%2C1%2C%22Z%22%5D%2C%5B1%2C1%2C1%2C1%2C%22Bloch%22%5D%2C%5B1%2C1%2C1%2C1%2C%22~f7c0%22%5D%5D%2C%22gates%22%3A%5B%7B%22id%22%3A%22~87lj%22%2C%22name%22%3A%22message%22%2C%22circuit%22%3A%7B%22cols%22%3A%5B%5B%22e%5E-iYt%22%5D%2C%5B%22X%5Et%22%5D%5D%7D%7D%2C%7B%22id%22%3A%22~f7c0%22%2C%22name%22%3A%22received%22%2C%22matrix%22%3A%22%7B%7B1%2C0%7D%2C%7B0%2C1%7D%7D%22%7D%5D%7D)

In [11]:
### -YY measurement
SSdgI = TensorProduct(S, S_dg, I)
HHI = TensorProduct(H, H, I)

# Transform measurement 
result = HHI * SSdgI * state_before_measurement

# Normalize and simplify the result
norm_factor = (1-sp.I)/4
final_state = sp.simplify(result / norm_factor)
print("System state (w/o coefficent):")
sp.pprint(final_state, use_unicode=True)
print()

# Extract and group the components by measuring qubits q0 and q1 
# |q0 q1 q2⟩ → |q0 q1⟩ ⊗ |q2⟩
components = [sp.simplify(final_state[i]) for i in range(8)]
grouped = {
    "00": [components[0], components[1]],
    "01": [components[2], components[3]],
    "10": [components[4], components[5]],
    "11": [components[6], components[7]]
}

# Restoration gates
default_gate = Z * H * S
restoration_gate_map = {
    "00": default_gate,
    "01": default_gate * X,
    "10": default_gate * Y,
    "11": default_gate * Y * X
}

# Normalization factors for assertion check
normalization_map = {
    "00": sp.sqrt(2) * sp.I,       
    "01": -sp.sqrt(2), 
    "10": -sp.sqrt(2),
    "11": -sp.sqrt(2) * sp.I         
}

for outcome, gate in restoration_gate_map.items():
    trans_state = gate * sp.Matrix(grouped[outcome])
    trans_state = sp.simplify(trans_state / normalization_map[outcome])
    assert input_q.equals(trans_state), f"[ERROR] {outcome}: {trans_state}"
    print(f"[PASS] -YY measurement for |q0 q1⟩ = {outcome}")

System state (w/o coefficent):
⎡ⅈ⋅(a - b) ⎤
⎢          ⎥
⎢  a + b   ⎥
⎢          ⎥
⎢ⅈ⋅(a + b) ⎥
⎢          ⎥
⎢  -a + b  ⎥
⎢          ⎥
⎢  a + b   ⎥
⎢          ⎥
⎢ⅈ⋅(-a + b)⎥
⎢          ⎥
⎢  a - b   ⎥
⎢          ⎥
⎣ⅈ⋅(a + b) ⎦

[PASS] -YY measurement for |q0 q1⟩ = 00
[PASS] -YY measurement for |q0 q1⟩ = 01
[PASS] -YY measurement for |q0 q1⟩ = 10
[PASS] -YY measurement for |q0 q1⟩ = 11
